# CI/CD for Agents (Regression Testing, Golden Datasets, and Continuous Improvement)
Building an autonomous agent prototype is relatively easy; keeping it working reliably as you update prompts, swap out underlying LLM checkpoints, or add new tools is exceptionally difficult. Without automated testing, every minor tweak runs the risk of breaking downstream reasoning paths—a phenomenon known as AI regression.

In this final topic of Module 09, we examine how to integrate agent evaluation into a Continuous Integration / Continuous Deployment (CI/CD) pipeline.

## 1. Why Traditional Software CI/CD Isn't Enough
In traditional software engineering, a CI/CD pipeline runs unit and integration tests every time code is pushed to GitHub. If all tests pass (exit code 0), the code merges and deploys.

[Git Push] ──► [Run Deterministic Unit Tests] ──► [100% Pass] ──► [Deploy to Production]


Why this breaks for Agents:

Stochastic Flakiness: Because LLMs are probabilistic, a test suite might pass 9 times out of 10 and fail on the 10th run due to a minor variation in token sampling.

Cost and Latency: Running an evaluation suite that includes LLM-as-a-Judge evaluations across 100 golden test cases takes minutes and costs real API tokens, unlike local milliseconds-fast unit tests.

Semantic Regressions: Code doesn't change, but a model provider updates their underlying weights (e.g., updating gemini-2.5-flash), silently altering the agent's behavior.

## 2. The Core Pillars of Agent CI/CD

To build a robust agent pipeline, your CI/CD architecture must incorporate four core components:

A. The Immutable Golden DatasetYour evaluation dataset must be version-controlled alongside your code (e.g., in a tests/golden_dataset.json file in your repository). Every time an agent fails in production or encounters a new edge case, a new test case is added to this dataset.

B. Tiered Test Suites (Fast vs. Deep Evals)To manage CI/CD execution time and token costs, split your tests into tiers:

Smoke Tests (Pre-Commit / PR Check): Runs a small subset (e.g., 5-10 core test cases) using deterministic checks (tool schemas, keyword matches) to catch glaring errors instantly.

Regression Suite (Nightly / Pre-Release): Runs the full golden dataset (100+ test cases) incorporating LLM-as-a-Judge scoring, multi-turn trajectory evals, and cost/latency tracking.

C. Threshold-Based Pass GatesBecause LLMs have inherent variance, CI/CD pipelines should not enforce a rigid 100% pass rate for semantic tests. Instead, set statistical thresholds:

Example: "The overall agent pass rate on the golden dataset must be $\ge 92\%$, and tool selection accuracy must be $100\%$." If a pull request drops the pass rate below $92\%$, the PR is automatically blocked.

D. Automated Drift and Cost TrackingYour CI/CD pipeline should track three key metrics over time on every commit:

Pass Rate % (Quality)Average Token Count / Cost per Run (Efficiency)Average Execution Latency / Steps (Speed)

## 3. Architecture of an Agent CI/CD Pipeline (GitHub Actions Example)

[Developer Push / PR] 
        │
        ▼
[GitHub Actions Runner]
        │
        ├── 1. Install Dependencies & SDKs
        ├── 2. Run Deterministic Unit Tests (Fast)
        └── 3. Run Agent Evaluation Harness (Golden Dataset + LLM Judge)
                    │
                    ▼
         [Evaluate Pass Rate Threshold (>= 92%)]
                    │
            ┌───────┴───────┐
         (Pass)          (Fail)
            │               │
            ▼               ▼
     [Merge Allowed]   [Block PR & Notify Team]


## 4. Sample GitHub Actions Workflow (agent_eval.yml)
Below is a production-style GitHub Actions YAML configuration that automates your agent evaluation suite on every pull request

In [ ]:
name: Agent Evaluation CI/CD

on:
  pull_request:
    branches: [ main ]
  push:
    branches: [ main ]

jobs:
  evaluate-agent:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout Code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          cache: 'pip'

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run Agent Evaluation Harness
        env:
          GEMINI_API_KEY: ${{ secrets.GEMINI_API_KEY }}
        run: |
          python scripts/run_eval_suite.py --threshold 92.0

# Key Takeaways
Treat Prompts as Code: Teach students that because prompts and tool definitions dictate agent behavior, they must be treated with the exact same rigor as source code—subject to pull requests, code reviews, and automated test gates.

Continuous Feedback Loop: Emphasize that an agentic system is never "finished." The CI/CD pipeline acts as the bridge between production failures and development improvements, turning every real-world bug into a permanent automated regression test.